# Appendix D：E4–E8 联合优化实验

保留原 `Nulling_CDF_SectorDrop.ipynb`；本入口替换 λ 控制为 robust SOCP + 完整历史 LP。
**主 CDF 固定 TN 需求/截止时间/失败预算，只扫描 Γ；不可行不会自动放宽 TN 条件。**

先读 [APPENDIX_D_EXPERIMENTS.md](APPENDIX_D_EXPERIMENTS.md)。
默认 quick 跑通所有图表；正式样本量将 QUICK 改为 False。有限模型结果与静态 Sionna 缓存结果分别标注，
不得将合成时间轨迹解释为已完成物理校准。原论文 CDF/radiomap 不会被替换。


In [ ]:
from pathlib import Path
import importlib.util, sys
ROOT = Path.cwd()
if not (ROOT / 'run_appendix_d.py').is_file():
    raise RuntimeError('请将 notebook 工作目录切换到项目根目录。')
missing = [p for p in ('numpy','scipy','matplotlib','cvxpy','clarabel')
           if importlib.util.find_spec(p) is None]
if missing:
    raise ImportError(f'当前 kernel 缺少 {missing}。请按 APPENDIX_D_EXPERIMENTS.md 创建独立环境，'
                      '或在当前 kernel 执行 %pip install -r requirements-appendix-d.txt。'
                      '为避免影响 Sionna，建议独立环境。')
from run_appendix_d import run_suite
from IPython.display import display, Markdown, Image, FileLinks
from datetime import datetime


## 实验配置

- Γ 越小，NTN 允许的干扰幅度越严格。
- THETA_DL/UL × 固定无空隙参考服务 = 截止期需求，不随 Γ 重算参考量。
- DELTA_TN 是逐方向/窗口允许失败概率；DELTA_NTN 是活动时间超标比例。
- 静态缓存的 −6 dB TN SNR 可行性标记与上述窗口 QoS 不是同一种约束。


In [ ]:
EXPERIMENTS = ['E4', 'E5', 'E6', 'E7', 'E8']
QUICK = True
SEED = 20260921
GAMMA_DB = [-15, -10, -5, 0]
THETA_DL, THETA_UL = 0.40, 0.50
DELTA_TN, DELTA_NTN = 0.10, 0.08
RUN_CACHED_SPATIAL = False
CACHE_DIR = ROOT / 'result/20260916_080614_095092'
CACHE_MAX_SECTORS = 4 if QUICK else None
OUT = ROOT / 'result' / ('appendix_d_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
print('输出目录:', OUT)


In [ ]:
run_suite(OUT, experiments=EXPERIMENTS, quick=QUICK, seed=SEED,
          gamma_db=GAMMA_DB, theta_dl=THETA_DL, theta_ul=THETA_UL,
          delta_tn=DELTA_TN, delta_ntn=DELTA_NTN,
          cache_dir=CACHE_DIR if RUN_CACHED_SPATIAL else None,
          cache_max_sectors=CACHE_MAX_SECTORS)


In [ ]:
def show_results(name):
    folder = OUT / name
    if not folder.exists():
        display(Markdown(f'{name} 未在本次运行中选择。'))
        return
    for figure in sorted(folder.glob('*.png')):
        display(Image(filename=str(figure), width=900))
    display(Markdown(f'**全部 PDF / CSV / LaTeX / JSON：** `{folder}`'))
    display(FileLinks(str(folder), included_suffixes=['.pdf', '.csv', '.tex', '.json']))


## E4

完整相干信道集合的校准与覆盖。独立整场景分割；区分已知有限界与经验边际覆盖，保留漏检和 UL 静默背景。

In [ ]:
show_results('E4')

## E5

精确最坏泄漏、SOCP、零预算零空间、功率回退、ρ/Γ/集合秩/方向夹角，以及旧方法和 oracle 对照。

In [ ]:
show_results('E5')

## E6

波束消去条件、完整确定性策略的随机混合枚举 vs 占用 LP、随机化必要性和不可行实例。

In [ ]:
show_results('E6')

## E7

F/T/L/J 同一模型下比较；固定 TN 条件的 Γ-CDF、TN SNR、净服务、时间轴和单独的 θ×Γ 可行域。

In [ ]:
show_results('E7')

## E8

失败与延迟、UL 静默背景、增益/观测核失配、合法 gap 缺失、新到达。失配实验不声称保持模型内保证。

In [ ]:
show_results('E8')

## 可选：原 Sionna 缓存的新 Γ-CDF

在配置中启用 RUN_CACHED_SPATIAL。仅静态单BS经验验证，包含 TN 不可行扇区，不能代替动态联合保证。

In [ ]:
show_results('cached_spatial')